In [ ]:
from snowflake.snowpark import Session
from snowflake.ml.modeling.impute import SimpleImputer
from snowflake.ml.modeling.preprocessing import StandardScaler
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.utils.connection_params import SnowflakeLoginOptions
from dotenv import load_dotenv
from snowflake.ml.modeling.metrics import accuracy_score
import os

In [ ]:
load_dotenv()

In [ ]:
os.environ.get('SNOWFLAKE_USER')

In [ ]:
session=Session.builder.configs(SnowflakeLoginOptions("my_example_connection")).create()
session

In [ ]:
session

In [ ]:
query="""
SELECT 
    *,
    IFF(CLASS='g',1.0,0.0) AS LABEL 
FROM 
    test.public.telescope
"""

In [ ]:
df=session.sql(query=query).drop('CLASS')

In [ ]:
df.show()

In [ ]:
train_data, test_data=df.random_split(weights=[0.9,0.1],seed=43)

In [ ]:
train_data.show()

In [ ]:
COLS=[c for c in train_data.columns if c!='LABEL']
COLS

In [ ]:
model=Pipeline(steps=[

    ('imputer', SimpleImputer(input_cols=COLS,output_cols=COLS)),
    ('scaler',StandardScaler(input_cols=COLS, output_cols=COLS)),
    ('clf',XGBClassifier(input_cols=COLS, label_cols=["LABEL"]))
])

In [ ]:
model.fit(train_data)

In [ ]:
train_data_pred=model.predict(train_data)

In [ ]:
train_data_pred.show()

In [ ]:
training_accuracy=accuracy_score(
    df=train_data_pred,
    y_true_col_names=['LABEL'],
    y_pred_col_names=['OUTPUT_LABEL']
)
training_accuracy

In [ ]:
test_data_pred=model.predict(test_data)
eval_accuracy=accuracy_score(
    df=test_data_pred,
    y_true_col_names=['LABEL'],
    y_pred_col_names=['OUTPUT_LABEL']
)
eval_accuracy